# Earth Terrain Features — Google Earth Engine (Colab)

**Input:** CSV containing `site_id, latitude_deg, longitude_deg` (or `site_id, lat, lon`).  
**Output:** `gee_terrain_features.csv` with `elevation_m`, `mean_slope_deg`, `terrain_roughness_m`.

- Source: Copernicus DEM GLO-30 (`COPERNICUS/DEM/GLO30`), ~30 m digital **surface** model.
- Mỗi site: 9 điểm lấy mẫu (3×3) với offset **0.0005 độ**. `elevation_m` và `mean_slope_deg` lấy ở điểm giữa; `terrain_roughness_m` = độ lệch chuẩn population (`ddof=0`) của **9** độ cao hợp lệ.
- `mean_slope_deg` giữ tên cột theo yêu cầu, nhưng thực chất là **slope tại tâm**, không phải trung bình slope của 9 điểm.
- Tất cả phép đọc raster và slope diễn ra trên máy chủ Earth Engine; Colab chỉ nhận bảng mẫu. Không cần download DEM.

**Prerequisites:** An authorized Earth Engine account and a registered Google Cloud Project. Xem [Colab setup](https://developers.google.com/earth-engine/guides/python_install-colab) và [dataset](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_DEM_GLO30).

**How to run:** Execute cells sequentially from top to bottom. Re-run relevant cells if you change input files or PROJECT_ID.
**Quy tắc dữ liệu thiếu:** CSV đầu ra giữ nguyên mọi dòng đầu vào; site lỗi ghi `N/A` cho cả 3 cột terrain.


In [ ]:
# Cell 1 — Imports (Colab usually pre-installs earthengine-api)
import sys
import subprocess
try:
    import ee
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'earthengine-api'])
    import ee

import io
import time
import numpy as np
import pandas as pd
from IPython.display import display
from google.colab import files
print('Earth Engine Python API:', ee.__version__)

## 1 — Earth Engine Authentication

Thay `YOUR_GCP_PROJECT_ID` bằng **Cloud Project ID** đã đăng ký Earth Engine (không phải tên hiển thị của project). Đăng nhập theo hướng dẫn nếu Colab yêu cầu.

In [ ]:
# Cell 2 — Authenticate and initialize
PROJECT_ID = 'YOUR_GCP_PROJECT_ID'  # <-- REQUIRED edit
if PROJECT_ID == 'YOUR_GCP_PROJECT_ID' or not PROJECT_ID.strip():
    raise ValueError('Please enter a valid Google Cloud PROJECT_ID registered with Earth Engine.')

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print('Earth Engine connected:', ee.String('OK').getInfo())

## 2 — Upload file CSV

Bấm **Choose files** để chọn CSV. Các dòng có tọa độ sai hoặc thiếu **vẫn được giữ lại**, các cột địa hình là `N/A`.


In [ ]:
# Cell 3 — Upload CSV; retain every input site, including invalid coordinates
DELTA_DEG = 0.0005

uploaded = files.upload()
if not uploaded:
    raise ValueError('No CSV file uploaded.')
filename, raw = next(iter(uploaded.items()))
input_df = pd.read_csv(io.BytesIO(raw), encoding='utf-8-sig')
input_df.columns = input_df.columns.astype(str).str.strip().str.lower()

def find_column(df, candidates, required=True):
    matches = [name for name in candidates if name in df.columns]
    if matches:
        return matches[0]
    if required:
        raise ValueError(f'Missing columns {candidates}. Available columns: {list(df.columns)}')
    return None

lat_col = find_column(input_df, ['latitude_deg', 'latitude', 'lat'])
lon_col = find_column(input_df, ['longitude_deg', 'longitude', 'lon', 'lng'])
id_col = find_column(input_df, ['site_id', 'site', 'id'], required=False)

# all_sites preserves 100% of input CSV rows and order.
all_sites = pd.DataFrame({
    'site_id': input_df[id_col].astype('string') if id_col else [f'SITE_{i+1:03d}' for i in range(len(input_df))],
    'latitude_deg': pd.to_numeric(input_df[lat_col], errors='coerce').astype('float64'),
    'longitude_deg': pd.to_numeric(input_df[lon_col], errors='coerce').astype('float64'),
})
missing_id = all_sites['site_id'].isna() | (all_sites['site_id'].str.strip() == '')
all_sites.loc[missing_id, 'site_id'] = [f'SITE_{i+1:03d}' for i in all_sites.index[missing_id]]
all_sites['site_index'] = np.arange(len(all_sites), dtype=np.int64)

valid = (
    np.isfinite(all_sites['latitude_deg']) & np.isfinite(all_sites['longitude_deg'])
    & all_sites['latitude_deg'].between(-90 + DELTA_DEG, 90 - DELTA_DEG)
    & all_sites['longitude_deg'].between(-180, 180)
)
invalid_sites = all_sites.loc[~valid].copy()
sites = all_sites.loc[valid].copy().reset_index(drop=True)

print(f'File: {filename} | Input: {len(all_sites)} | Valid: {len(sites)} | Invalid: {len(invalid_sites)}')
if len(invalid_sites):
    print('Sites with invalid coordinates remain in the output CSV with terrain = N/A.')
    display(invalid_sites.head(10))
display(all_sites[['site_id', 'latitude_deg', 'longitude_deg']].head())


## 3 — Build DEM & Slope at Native Projection

Sau `mosaic()`, đặt phép chiếu mặc định từ tile gốc trước khi gọi `ee.Terrain.slope()`. Earth Engine khuyến cáo thao tác này để tránh mosaic bị gán phép chiếu mặc định thô (1°).

In [ ]:
# Cell 4 — GLO-30 DEM and center-pixel slope
DEM_ASSET = 'COPERNICUS/DEM/GLO30'
collection = ee.ImageCollection(DEM_ASSET).select('DEM')
native_projection = collection.first().projection()
dem = collection.mosaic().setDefaultProjection(native_projection).rename('elevation_m')
slope = ee.Terrain.slope(dem).rename('slope_deg')
# Only require valid slope at the center point: unmask other slopes so the 9 elevation samples are preserved.
terrain_image = dem.addBands(slope.unmask(-9999))
print('DEM:', DEM_ASSET)
print('Native projection:', native_projection.getInfo()['crs'])

## 4 — Batch Sampling, Aggregation & CSV Download

Kết quả luôn giữ **đúng số dòng và thứ tự input**, kể cả site lỗi/batch lỗi. Cột terrain của các site không đủ 9 elevation hoặc slope tại tâm sẽ ghi `N/A`. File lỗi riêng ghi nguyên nhân. Cột số vẫn là `float64` trong pandas.


In [ ]:
# Cell 5 — Extract terrain; KEEP every input site, mark missing terrain as N/A
SITES_PER_BATCH = 10
MAX_RETRIES = 3
RETRY_BASE_SECONDS = 2.0
SAMPLE_SCALE_M = 30
OUTPUT_CSV = 'gee_terrain_features.csv'
FAILED_CSV = 'gee_terrain_failed_sites.csv'

TERRAIN_COLUMNS = ['elevation_m', 'mean_slope_deg', 'terrain_roughness_m']
OUTPUT_COLUMNS = ['site_id', 'latitude_deg', 'longitude_deg', *TERRAIN_COLUMNS]

def wrap_longitude(lon):
    return ((float(lon) + 180) % 360) - 180

def make_grid_features(part):
    points = []
    for row in part.itertuples(index=False):
        for dy_idx in (-1, 0, 1):
            for dx_idx in (-1, 0, 1):
                lat = float(row.latitude_deg) + dy_idx * DELTA_DEG
                lon = wrap_longitude(float(row.longitude_deg) + dx_idx * DELTA_DEG)
                points.append(ee.Feature(ee.Geometry.Point([lon, lat]), {
                    'site_index': int(row.site_index),
                    'grid_dy': dy_idx,
                    'grid_dx': dx_idx,
                }))
    return ee.FeatureCollection(points)

def fetch_batch(part):
    sampled = terrain_image.sampleRegions(
        collection=make_grid_features(part),
        properties=['site_index', 'grid_dy', 'grid_dx'],
        scale=SAMPLE_SCALE_M,
        geometries=False,
        tileScale=4,
    )
    return sampled.getInfo().get('features', [])

def interpret_batch(part, fetched):
    by_site = {}
    for feature in fetched:
        props = feature.get('properties', {})
        try:
            idx = int(props['site_index'])
            dy = int(props['grid_dy'])
            dx = int(props['grid_dx'])
            elev = float(props['elevation_m'])
        except (KeyError, TypeError, ValueError):
            continue
        if np.isfinite(elev):
            by_site.setdefault(idx, {})[(dy, dx)] = {
                'elevation': elev,
                'slope': props.get('slope_deg'),
            }

    passed, failed = [], []
    expected = {(dy, dx) for dy in (-1, 0, 1) for dx in (-1, 0, 1)}
    for row in part.itertuples(index=False):
        pts = by_site.get(int(row.site_index), {})
        base = {'site_index': int(row.site_index), 'site_id': str(row.site_id),
                'latitude_deg': float(row.latitude_deg), 'longitude_deg': float(row.longitude_deg)}
        if set(pts) != expected:
            failed.append({**base, 'reason': f'missing_elevation_points_{9-len(pts)}'})
            continue
        try:
            center_slope = float(pts[(0, 0)]['slope'])
        except (TypeError, ValueError, KeyError):
            center_slope = np.nan
        if not np.isfinite(center_slope) or center_slope < 0:
            failed.append({**base, 'reason': 'missing_center_slope'})
            continue
        elevations = np.asarray([pts[k]['elevation'] for k in sorted(expected)], dtype=np.float64)
        passed.append({
            'site_index': int(row.site_index),
            'elevation_m': np.float64(pts[(0, 0)]['elevation']),
            'mean_slope_deg': np.float64(center_slope),
            'terrain_roughness_m': np.float64(np.std(elevations, ddof=0)),
        })
    return passed, failed

all_rows = []
all_failed = [
    {'site_index': int(r.site_index), 'site_id': str(r.site_id),
     'latitude_deg': r.latitude_deg, 'longitude_deg': r.longitude_deg,
     'reason': 'invalid_coordinates'}
    for r in invalid_sites.itertuples(index=False)
]
num_batches = (len(sites) + SITES_PER_BATCH - 1) // SITES_PER_BATCH
for start in range(0, len(sites), SITES_PER_BATCH):
    part = sites.iloc[start:start + SITES_PER_BATCH]
    batch_num = start // SITES_PER_BATCH + 1
    fetched, last_error = None, None
    for attempt in range(MAX_RETRIES):
        try:
            fetched = fetch_batch(part)
            break
        except Exception as exc:
            last_error = exc
            if attempt < MAX_RETRIES - 1:
                delay = RETRY_BASE_SECONDS * (2 ** attempt)
                print(f'Batch {batch_num}/{num_batches} lỗi ({type(exc).__name__}). Retry sau {delay:.0f}s...')
                time.sleep(delay)
    if fetched is None:
        print(f'Batch {batch_num}/{num_batches} failed: {str(last_error)[:180]}')
        for row in part.itertuples(index=False):
            all_failed.append({
                'site_index': int(row.site_index), 'site_id': str(row.site_id),
                'latitude_deg': float(row.latitude_deg), 'longitude_deg': float(row.longitude_deg),
                'reason': f'earth_engine_batch_error: {str(last_error)[:140]}',
            })
        continue
    passed, failed = interpret_batch(part, fetched)
    all_rows.extend(passed)
    all_failed.extend(failed)
    print(f'Batch {batch_num}/{num_batches}: {len(passed)} OK, {len(failed)} N/A')

# Merge by original row position rather than site_id (which may contain duplicates).
# This guarantees each input CSV row maps to exactly one output CSV row.
result = all_sites[['site_index', 'site_id', 'latitude_deg', 'longitude_deg']].copy()
for col in TERRAIN_COLUMNS:
    result[col] = np.nan
if all_rows:
    success_df = pd.DataFrame(all_rows).set_index('site_index')
    for col in TERRAIN_COLUMNS:
        result.loc[result['site_index'].isin(success_df.index), col] = (
            result.loc[result['site_index'].isin(success_df.index), 'site_index'].map(success_df[col]).to_numpy()
        )
result = result[OUTPUT_COLUMNS]
for col in ['latitude_deg', 'longitude_deg', *TERRAIN_COLUMNS]:
    result[col] = pd.to_numeric(result[col], errors='coerce').astype('float64')
result[TERRAIN_COLUMNS] = result[TERRAIN_COLUMNS].replace([np.inf, -np.inf], np.nan)

# na_rep='N/A' only changes CSV serialization; numeric columns stay float64 in pandas.
result.to_csv(OUTPUT_CSV, index=False, float_format='%.8f', na_rep='N/A')
print(f'Saved {len(result)}/{len(all_sites)} sites (all rows preserved): {OUTPUT_CSV}')
print(f'Succeeded: {len(all_rows)} | Terrain N/A: {len(all_sites) - len(all_rows)}')
display(result.head(10))
files.download(OUTPUT_CSV)

if all_failed:
    failure_report = pd.DataFrame(all_failed).sort_values('site_index').drop(columns=['site_index'])
    failure_report.to_csv(FAILED_CSV, index=False, na_rep='N/A')
    print(f'Failed/missing sites: {len(failure_report)}. Report: {FAILED_CSV}')
    display(failure_report.head(10))
    files.download(FAILED_CSV)
else:
    print('No failed sites.')


## Scientific Notes & Troubleshooting

1. **Slope**: `ee.Terrain.slope()` ước lượng slope (độ) từ các pixel DEM lân cận. Vì chỉ lấy slope tại tâm, `mean_slope_deg` **không phải** phép trung bình slope trên toàn bộ ô 3×3.
2. **Roughness**: độ lệch chuẩn 9 mẫu có thể nhạy với vị trí pixel và vĩ độ; không nên so sánh trực tiếp mọi vĩ độ nếu cần cùng footprint vật lý. Nếu cần khoảng cách cố định theo mét, phải thay cách tạo grid.
3. **DSM, không phải bare-earth DTM**: Copernicus GLO-30 có thể chứa chiều cao tán cây / công trình.
4. **Không hoàn toàn toàn cầu**: dữ liệu mặt đất bị mask ở một số nơi; một site thiếu điểm vẫn xuất hiện trong file ML với terrain `N/A`. Xem `gee_terrain_failed_sites.csv`.
5. **Lỗi quyền truy cập**: xác minh Cloud Project, đăng ký Earth Engine và đăng nhập. Batch retry không khắc phục lỗi permission/project.
6. **Lỗi memory / computation limits**: giảm `SITES_PER_BATCH` từ 10 xuống 5 hoặc 2. Các tọa độ khác vùng địa lý vẫn dùng chung server-side mosaic nên phép tính có thể tốn thời gian.
7. **Dữ liệu nâng cao**: nếu cần độ dốc trung bình thực sự của vùng lân cận, thay slope tại tâm bằng phép lấy trung bình slope theo vùng (một định nghĩa khác).

Reference: [Copernicus DEM GLO-30](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_DEM_GLO30), [Terrain slope example](https://developers.google.com/earth-engine/apidocs/ee-terrain-slope), [GEE Python Colab setup](https://developers.google.com/earth-engine/guides/python_install-colab).